In [0]:
# Notebook autonome : il porte ses propres %run et se lance seul.
# Relancés par main_translations, ils sont sans effet de bord (idempotents).

In [0]:
%run ./env

In [0]:
%run ./python_libraries

In [0]:
%run ../delta_function

In [0]:
%run ./translation_function

In [0]:
%run ./dim_trad_language

# build_translations

Construit les tables de traduction du modèle à partir des tables
`*_translations` de la base PostgreSQL.

## Pourquoi une transformation est nécessaire

Les tables source ne peuvent pas être branchées telles quelles sur le modèle :

1. **Le fallback demandé par le PO** (langue -> anglais -> la clé, jamais de null)
   n'est pas réalisable en RLS. Le RLS sait seulement *supprimer* des lignes, pas
   en substituer une autre. Le repli doit donc être matérialisé ici.
2. **Les traductions sont incomplètes** : toutes les langues ne sont pas
   renseignées pour toutes les clés. Sans traitement, une clé non traduite
   *disparaîtrait* du visuel après filtrage RLS — les lignes seraient absentes et
   **les totaux seraient faux**, ce qui est bien plus grave qu'un libellé anglais.
3. **Le couple (clé, langue) doit être unique et toujours présent** pour qu'après
   filtrage RLS il reste exactement une ligne par clé. Sinon les visuels
   dupliquent les lignes.

On produit donc des tables **denses** : le produit cartésien
`clés x langues actives`, avec un libellé garanti non nul sur chaque ligne.

## Chargement des sources

Ce notebook ne dépend pas de `load_data` : il tourne dans un job autonome qui
n'a pas besoin des tables de mesures. Il charge donc directement les 8 tables de
traduction depuis `source_catalog`, défini dans `env`.

In [0]:
goods_species_translations = spark.table(f"{source_catalog}.goods_species_translations")
goods_varieties_translations = spark.table(f"{source_catalog}.goods_varieties_translations")
parameters_production_type_translations = spark.table(f"{source_catalog}.parameters_production_type_translations")
parameters_variables_translations = spark.table(f"{source_catalog}.parameters_variables_translations")
parameters_production_line_variables_translations = spark.table(f"{source_catalog}.parameters_production_line_variables_translations")
parameters_localizations_translations = spark.table(f"{source_catalog}.parameters_localizations_translations")
parameters_localization_groups_translations = spark.table(f"{source_catalog}.parameters_localization_groups_translations")
parameters_batch_note_categories_translations = spark.table(f"{source_catalog}.parameters_batch_note_categories_translations")

## Traductions à clé simple

Ces cinq tables partagent le même patron : une clé entière, une langue, un
libellé. La clé est renommée pour correspondre à la colonne portée par la table
du modèle qui s'y rattachera.

## Les clés viennent des nomenclatures

La règle du PO — *« langue demandée, sinon anglais, sinon la clé ; jamais de
null »* — ne peut s'appliquer qu'à une ligne qui existe. Si les clés étaient
tirées des tables `*_translations`, une clé jamais traduite dans aucune langue
n'aurait aucune ligne, et son libellé s'afficherait **vide** dans le rapport :
le repli ne se déclencherait pas.

Les clés sont donc lues dans les nomenclatures de `common`, qui portent la liste
complète de ce qui existe réellement.

**Le repli de dernier recours est le `code` métier de la nomenclature**
(`ORGE_6RH`, `TCR`…) et non l'identifiant technique (`42`, `detail_TCR`) :
c'est l'interprétation la plus vraisemblable de « la clé » dans la bouche du PO,
et la seule qui produise quelque chose de lisible.

> Ce notebook doit s'exécuter **après** `build_nomenclatures`.
>
> Deux points restent à confirmer avec le PO : ce qu'il entend exactement par
> « la clé », et s'il a connaissance des clés jamais traduites (127 catégories
> de détail sur 397 en preprd).

In [0]:
def cles_nomenclature(table, key_col, alias, label_col=None):
    """Clés actives d'une nomenclature, renommées comme dans la table de traduction.

    label_col : la colonne de code métier, qui servira de repli de dernier recours.
    Les nomenclatures conservent les lignes supprimées (drapeau deleted) : on les
    écarte ici, une entité supprimée n'ayant pas à être traduite.
    """
    cols = [F.col(key_col).alias(alias)]
    if label_col is not None:
        cols.append(F.col(label_col).cast("string").alias(FALLBACK_LABEL_COL))
    else:
        # Pas de code métier dans cette nomenclature : on retombe sur l'identifiant.
        cols.append(F.col(key_col).cast("string").alias(FALLBACK_LABEL_COL))

    return (
        spark.table(f"{current_catalog}.{common_schema}.{table}")
        .filter(F.col("deleted") == False)
        .select(*cols)
    )

In [0]:
# goods_species_translations -> dim_trad_specy
dim_trad_specy = build_translation_dim(
    goods_species_translations,
    dim_trad_language,
    keys_df=cles_nomenclature("dim_specy", "id_good_specy", "good_specy", "code"),
    key_cols=["good_specy"]
).withColumnRenamed("good_specy", "id_good_specy")

publish_dim(dim_trad_specy, "dim_trad_specy", ["id_good_specy", "language"], current_catalog + "." + common_schema)

In [0]:
# goods_varieties_translations -> dim_trad_variety
dim_trad_variety = build_translation_dim(
    goods_varieties_translations,
    dim_trad_language,
    keys_df=cles_nomenclature("dim_variety", "id_good_variety", "good_variety", "code"),
    key_cols=["good_variety"]
).withColumnRenamed("good_variety", "id_good_variety")

publish_dim(dim_trad_variety, "dim_trad_variety", ["id_good_variety", "language"], current_catalog + "." + common_schema)

In [0]:
# parameters_production_type_translations -> dim_trad_production_type
# La source nomme sa clé "id_parameters_production_type" (pluriel) alors que la
# table métier parameters_production_types porte "id_parameter_production_type"
# (singulier). On normalise ici sur le nom singulier, celui du modèle.
dim_trad_production_type = build_translation_dim(
    parameters_production_type_translations,
    dim_trad_language,
    keys_df=cles_nomenclature("dim_production_type", "id_parameter_production_type", "id_parameters_production_type", "code"),
    key_cols=["id_parameters_production_type"]
).withColumnRenamed("id_parameters_production_type", "id_parameter_production_type")

publish_dim(
    dim_trad_production_type, "dim_trad_production_type",
    ["id_parameter_production_type", "language"], current_catalog + "." + common_schema)

In [0]:
# parameters_variables_translations -> dim_trad_variable
dim_trad_variable = build_translation_dim(
    parameters_variables_translations,
    dim_trad_language,
    keys_df=cles_nomenclature("dim_variable", "id_parameter_variable", "parameter_variable", "code"),
    key_cols=["parameter_variable"]
).withColumnRenamed("parameter_variable", "id_parameter_variable")

publish_dim(
    dim_trad_variable, "dim_trad_variable", ["id_parameter_variable", "language"], current_catalog + "." + common_schema)

In [0]:
# parameters_production_line_variables_translations -> dim_trad_production_line_variable
dim_trad_production_line_variable = build_translation_dim(
    parameters_production_line_variables_translations,
    dim_trad_language,
    keys_df=cles_nomenclature("dim_production_line_variable", "id_parameter_production_line_variable", "parameter_production_line_variable", None),
    key_cols=["parameter_production_line_variable"]
).withColumnRenamed(
    "parameter_production_line_variable", "id_parameter_production_line_variable"
)

publish_dim(
    dim_trad_production_line_variable, "dim_trad_production_line_variable",
    ["id_parameter_production_line_variable", "language"], current_catalog + "." + common_schema)

In [0]:
# parameters_localizations_translations -> dim_trad_localization
dim_trad_localization = build_translation_dim(
    parameters_localizations_translations,
    dim_trad_language,
    keys_df=cles_nomenclature("dim_localization", "id_parameter_localization", "id_parameter_localization", "code"),
    key_cols=["id_parameter_localization"]
)

publish_dim(
    dim_trad_localization, "dim_trad_localization",
    ["id_parameter_localization", "language"], current_catalog + "." + common_schema)

## Groupes de localisation — clé ramenée à une seule colonne

`parameters_localization_groups_translations` porte une clé double
(`id_parameter_localization_group`, `production_line`) : le modèle *permet* qu'un
groupe ait un libellé différent selon la ligne de production.

**Vérifié en PROD : cette possibilité n'est pas utilisée.** Aucun couple
(groupe, langue) n'a plus d'un libellé distinct. On ramène donc la clé à
`id_parameter_localization_group` seul, ce qui aligne la table sur la
nomenclature `dim_localization_group` et donne une relation 1→\* propre dans
Power BI.

La cellule suivante contrôle cette hypothèse à chaque run : si le front se met
un jour à différencier les libellés par ligne de production, le job s'arrêtera
au lieu d'en retenir un au hasard.

In [0]:
# Contrôle : l'hypothèse "un seul libellé par (groupe, langue)" tient-elle ?
conflits_groupes = (
    parameters_localization_groups_translations
    .filter(F.col("deleted") == False)
    .groupBy("id_parameter_localization_group", "language")
    .agg(F.countDistinct("label").alias("nb_libelles"))
    .filter(F.col("nb_libelles") > 1)
)

if conflits_groupes.count() > 0:
    display(conflits_groupes)
    raise ValueError(
        "Des groupes de localisation ont plusieurs libellés selon la ligne de "
        "production. La clé de dim_trad_localization_group doit alors inclure "
        "production_line, et le modèle Power BI une colonne de substitution."
    )


dim_trad_localization_group = build_translation_dim(
    parameters_localization_groups_translations,
    dim_trad_language,
    keys_df=cles_nomenclature("dim_localization_group", "id_parameter_localization_group", "id_parameter_localization_group", "code"),
    key_cols=["id_parameter_localization_group"]
)

publish_dim(
    dim_trad_localization_group, "dim_trad_localization_group",
    ["id_parameter_localization_group", "language"], current_catalog + "." + common_schema)

## Traductions des notes de production

`parameters_batch_note_categories_translations` sert **quatre usages** dans une
seule table : la clé `batch_note_category` est préfixée par le type
(`location_...`, `event_...`, `detail_...`, `impact_...`).

On en produit **quatre tables distinctes** plutôt qu'une seule. Raison :
`fact_batch_note` porte quatre colonnes à traduire (location, event, detail,
impact), et Power BI n'autorise qu'**une seule relation active** entre deux
tables. Une table unique obligerait à trois relations inactives et à des
`USERELATIONSHIP` dans chaque mesure — ingérable sur des colonnes posées
directement sur les axes des visuels.

Le repli de niveau 3 affiche le code **sans son préfixe** (`TCR` et non
`detail_TCR`) : c'est ce que l'utilisateur reconnaît.

In [0]:
# Le type de catégorie vient de la nomenclature (category_class), source faisant
# autorité, plutôt que d'un découpage du préfixe de la clé.
# La nomenclature conserve les noms source : sa clé est id_batch_note_category.
batch_note_categories = (
    spark.table(f"{source_catalog}.parameters_batch_note_categories")
    .filter(F.col("deleted") == False)
    .select(
        F.col("id_batch_note_category").alias("batch_note_category"),
        F.col("category_class"),
        F.col("category_label"),
    )
)

batch_note_trad_base = (
    parameters_batch_note_categories_translations.alias("t")
    .join(
        F.broadcast(batch_note_categories).alias("n"),
        F.col("t.batch_note_category") == F.col("n.batch_note_category"),
        "inner"   # inner : une clé absente de la nomenclature n'a rien à traduire
    )
    .select("t.*", "n.category_class", "n.category_label")
)

if verbose_mode == 'debug':
    print("Traductions par type de catégorie :")
    display(
        batch_note_trad_base.filter(F.col("deleted") == False)
                            .groupBy("category_class").count().orderBy(F.desc("count"))
    )

In [0]:
BATCH_NOTE_TYPES = ["location", "event", "detail", "impact"]

for category_type in BATCH_NOTE_TYPES:
    df_type = batch_note_trad_base.filter(F.col("category_class") == category_type)

    dim_type = build_translation_dim(
        df_type,
        dim_trad_language,
        keys_df=cles_nomenclature(
            f"dim_batch_note_{category_type}",
            "id_batch_note_category", "batch_note_category",
            "category_label",
        ),
        key_cols=["batch_note_category"]
    )

    process_name = f"dim_trad_batch_note_{category_type}"
    globals()[process_name] = dim_type
    publish_dim(dim_type, process_name, ["batch_note_category", "language"], current_catalog + "." + common_schema)

## Contrôle qualité du rafraîchissement

`label_source` mesure la couverture réelle des traductions. Un taux élevé de
`fallback_key` sur une langue signale que le front n'a pas alimenté la table :
le rapport reste fonctionnel mais s'affiche en anglais ou en codes techniques.
C'est l'indicateur à remonter au PO, pas un incident technique.

In [0]:
# Le contrôle relit les tables Delta qui viennent d'être écrites, il ne rejoue pas
# les DataFrames. Deux raisons :
#   - performance : les DataFrames ne sont pas en cache, les réutiliser ferait
#     recalculer toute la chaîne depuis PostgreSQL (dédoublonnage, crossJoin,
#     jointures) et doublerait le temps du notebook ;
#   - fiabilité : on mesure ce qui est réellement en base, pas ce qui aurait dû
#     y être écrit.
couverture = reduce(
    lambda a, b: a.unionByName(b),
    [
        spark.table(target)
             .groupBy("language", "label_source")
             .count()
             .withColumn("table", F.lit(nom))
        for nom, target in tables_publiees
    ]
)

display(
    couverture.alias("c")
              .join(F.broadcast(dim_trad_language).alias("l"), "language", "left")
              .groupBy("table", "code")
              .pivot("label_source", ["translated", "fallback_en", "fallback_key"])
              .agg(F.sum("count"))
              .orderBy("table", "code")
)

## Audit : les clés jamais traduites

Elles sont désormais **présentes** dans le modèle, avec leur code métier en
repli — l'utilisateur ne voit plus de vide. Mais ce sont bien des traductions
manquantes, à remonter à l'équipe front.

`label_source = 'fallback_key'` les identifie : ni la langue demandée, ni
l'anglais. Une clé dans ce cas y est pour les quatre langues, c'est donc une
absence totale et non un trou ponctuel.

In [0]:
jamais_traduites = reduce(
    lambda a, b: a.unionByName(b),
    [
        spark.table(target)
             .filter(F.col("label_source") == "fallback_key")
             .select(
                 F.lit(nom).alias("table"),
                 F.col("label").alias("code_affiche"),
             )
             .distinct()
        for nom, target in tables_publiees
    ]
)
jamais_traduites.cache()

print("Clés sans aucune traduction, par table :")
display(jamais_traduites.groupBy("table").count().orderBy(F.desc("count")))

print("Détail, à transmettre à l'équipe front :")
display(jamais_traduites.orderBy("table", "code_affiche"))